# Lab 8A: Workflow patterns
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 8 · about 30 minutes**

**Objectives**
- Build prompt chaining, routing, parallelization, and evaluator-optimizer with the Messages API
- Log calls, tokens, and time for each pattern
- Decide which pattern fits which task

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

## A small instrumented call helper
Every pattern below uses it, so the comparison table is fair.

In [ ]:
LOG = []
def call(prompt, system=None, model=None, max_tokens=600, pattern=""):
    kw = {"system": system} if system else {}
    t0 = time.perf_counter()
    r = client.messages.create(model=model or config.MODEL, max_tokens=max_tokens, messages=[{"role": "user", "content": prompt}], **kw)
    LOG.append({"pattern": pattern, "seconds": time.perf_counter() - t0, "in": r.usage.input_tokens, "out": r.usage.output_tokens})
    return text_of(r)
REPORT = """Q3 network report (synthetic): Outage minutes fell from 410 to 260. Ticket volume rose 12%, mostly billing questions after the price change.
First-contact resolution was 71% (target 75%). Fiber installs grew 18%. Two regions missed repair targets: North (82%) and Coast (79%)."""

## 1. Prompt chaining: extract, then analyze, then format

In [ ]:
facts = call(f"List every metric in this report as 'metric: value' lines, nothing else.\n{REPORT}", pattern="chain")
analysis = call(f"From these metrics, identify the two biggest risks and one win, with the numbers.\n{facts}", pattern="chain")
summary = call(f"Write a 60-word executive summary from this analysis. Plain language.\n{analysis}", pattern="chain")
print(summary)

Each step is small and testable. If the summary is wrong, you can see which step went wrong.

## 2. Routing: classify, then send to a specialized prompt (and model)

In [ ]:
ROUTES = {"billing": ("You are a billing specialist. Explain charges plainly and offer the next step.", config.FAST_MODEL),
          "technical": ("You are a network technician. Give numbered troubleshooting steps.", config.MODEL)}
def route(ticket):
    label = call(f"Classify as billing or technical. Reply with one word.\n{ticket}", model=config.FAST_MODEL, max_tokens=5, pattern="route").strip().lower()
    system, model = ROUTES.get(label, ROUTES["technical"])
    return label, call(ticket, system=system, model=model, max_tokens=300, pattern="route")
for t in ["Why is my bill 20 USD higher this month?", "My router loses connection every evening."]:
    label, reply = route(t); print(f"[{label}] {reply[:160]}\n")

## 3. Parallelization: independent sections at the same time

In [ ]:
import concurrent.futures
SECTIONS = {"billing risk": "the billing ticket increase", "repair risk": "the regions that missed repair targets", "growth": "fiber install growth"}
def review(name_topic):
    name, topic = name_topic
    return name, call(f"In two sentences, assess {topic} in this report.\n{REPORT}", max_tokens=150, pattern="parallel")
t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
    parallel = dict(pool.map(review, SECTIONS.items()))
t_parallel = time.perf_counter() - t0
t0 = time.perf_counter(); sequential = dict(review(x) for x in SECTIONS.items()); t_seq = time.perf_counter() - t0
print(f"parallel {t_parallel:.1f}s vs sequential {t_seq:.1f}s")
for k, v in parallel.items(): print(f"- {k}: {v}")

## 4. Evaluator-optimizer: draft, critique, revise (capped)

In [ ]:
CRITERIA = ["under 70 words", "mentions both regions that missed targets", "includes one recommended action", "no jargon"]
CHECK = {"type": "object", "properties": {"passes": {"type": "boolean"}, "missing": {"type": "array", "items": {"type": "string"}}}, "required": ["passes", "missing"], "additionalProperties": False}
draft = call(f"Write a manager update on repair performance from this report.\n{REPORT}", max_tokens=250, pattern="evaluator")
for round_no in range(1, 4):                                  # cap: at most 3 rounds
    r = client.messages.create(model=config.FAST_MODEL, max_tokens=200, output_config={"format": {"type": "json_schema", "schema": CHECK}},
                               messages=[{"role": "user", "content": f"Criteria: {CRITERIA}\nText:\n{draft}\nDoes it meet every criterion?"}])
    verdict = json.loads(text_of(r)); LOG.append({"pattern": "evaluator", "seconds": 0, "in": r.usage.input_tokens, "out": r.usage.output_tokens})
    print(f"round {round_no}: passes={verdict['passes']} missing={verdict['missing']}")
    if verdict["passes"]: break
    draft = call(f"Revise to fix: {verdict['missing']}. Keep everything else.\n{draft}", max_tokens=250, pattern="evaluator")
print("\nFINAL:", draft)

## 5. Compare the patterns

In [ ]:
pd.DataFrame(LOG).groupby("pattern").agg(calls=("in", "size"), input_tokens=("in", "sum"), output_tokens=("out", "sum"), seconds=("seconds", "sum")).round(1)

## You should now have
- [ ] Four working pattern functions
- [ ] A comparison of calls, tokens, and time
- [ ] Evidence that parallel sections finish faster

## Check yourself
1. Which pattern would you use for your own feature, and why?
2. When does routing save money, and when does it add cost?
3. Why cap the evaluator loop?